In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('creditcard.csv')
print(df.head())

print(df.isnull().sum())



   Time        V1        V2        V3  ...       V27       V28  Amount  Class
0   0.0 -1.359807 -0.072781  2.536347  ...  0.133558 -0.021053  149.62      0
1   0.0  1.191857  0.266151  0.166480  ... -0.008983  0.014724    2.69      0
2   1.0 -1.358354 -1.340163  1.773209  ... -0.055353 -0.059752  378.66      0
3   1.0 -0.966272 -0.185226  1.792993  ...  0.062723  0.061458  123.50      0
4   2.0 -1.158233  0.877737  1.548718  ...  0.219422  0.215153   69.99      0

[5 rows x 31 columns]
Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64


In [4]:
X = df.drop('Class',axis=1)
y = df['Class']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.33,random_state=42)



y_train.value_counts()

Class
0    190477
1       343
Name: count, dtype: int64

In [5]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_bal,y_train_bal = smote.fit_resample(X_train,y_train)

print(y_train.value_counts())
print(y_train_bal.value_counts())

2026/05/27 12:39:36 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '1e4fcb4129b74bbbacdefc9f0806f549', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


2026/05/27 12:39:36 WARNING mlflow.sklearn: Failed to infer model signature: the trained model does not have a `predict` or `transform` function, which is required in order to infer the signature
2026/05/27 12:39:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/27 12:39:36 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2026/05/27 12:39:38 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels were not specified. To automatically record training metrics, provide training labels as inputs to the model training function.


Class
0    190477
1       343
Name: count, dtype: int64
Class
0    190477
1    190477
Name: count, dtype: int64


In [6]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    n_jobs=-1,
    random_state=42,
    class_weight='balanced'
)

rf_model.fit(X_train_bal,y_train_bal)

predictions = rf_model.predict(X_test)  
predictions_proba = rf_model.predict_proba(X_test)[:,1]

print(classification_report(y_test,predictions))
print(f"ROC-ACC score: {roc_auc_score(y_test,predictions_proba)}")

# Confusion Matrixxxx
cm = confusion_matrix(y_test,predictions)

print('CM')
print(f"True Negatives: ", {cm[0,0]})
print(f"False Poistives: ", {cm[0,1]})
print(f"False Negatives: ", {cm[1,0]})
print(f"True Positive: ", {cm[1,1]})

2026/05/27 12:39:39 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '63414285f8914fc097653d3626f49ecd', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/27 12:40:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     93838
           1       0.58      0.89      0.70       149

    accuracy                           1.00     93987
   macro avg       0.79      0.94      0.85     93987
weighted avg       1.00      1.00      1.00     93987

ROC-ACC score: 0.984289896438686
CM
True Negatives:  {np.int64(93741)}
False Poistives:  {np.int64(97)}
False Negatives:  {np.int64(17)}
True Positive:  {np.int64(132)}


In [21]:
pipe = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('xgb_model', XGBClassifier(
        n_estimators = 500,
        max_depth = 5,
        learning_rate = 0.2,
        random_state=42,
        min_child_weight =10,
        device="cuda"))
])

pipe.fit(X_train_bal,y_train_bal)

predictions_xgb = pipe.predict(X_test)
predictions_xgb_proba = pipe.predict_proba(X_test)[:,1]

threshold = 0.7

predictions_xgb_adj = (predictions_xgb_proba >= threshold).astype(int)

print(classification_report(y_test,predictions_xgb_adj))
print(f"ROC_ACC score:  {roc_auc_score(y_test,predictions_xgb_proba):.4f}")

#CM Matrix
cm_xgb = confusion_matrix(y_test,predictions_xgb)

print("confusion_matrix")
print(cm_xgb)


2026/05/27 13:35:19 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '5ad9140bc95745978297af4cf1217288', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/27 13:35:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     93838
           1       0.82      0.85      0.83       149

    accuracy                           1.00     93987
   macro avg       0.91      0.92      0.92     93987
weighted avg       1.00      1.00      1.00     93987

ROC_ACC score:  0.9825
confusion_matrix
[[93803    35]
 [   22   127]]


In [8]:
import joblib

exp = joblib.dump(pipe,'saved_model.pkl')

In [ ]:
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score,ConfusionMatrixDisplay,make_scorer
from sklearn.model_selection import GridSearchCV

mlflow.sklearn.autolog()


mlflow.set_experiment("Credit_card_fraud_detection")

with mlflow.start_run(run_name="XGboost_test"):

   
    pipe.fit(X_train_bal,y_train_bal)
    predictions_test = pipe.predict(X_test)
    
    predictions_proba_test = pipe.predict_proba(X_test)[:,1]
    roc_score_test = roc_auc_score(y_test,predictions_proba_test)
    report = classification_report(y_test,predictions_test)
    cm_test = ConfusionMatrixDisplay.from_predictions(y_test,predictions_test)


    mlflow.log_metric("roc_auc",roc_score_test)
    
    plt.savefig("confusion_matrix.png")

    mlflow.log_artifact("confusion_matrix.png")




    mlflow.sklearn.log_model(sk_model=pipe,
                             artifact_path="model",
                             registered_model_name="CreditCardFraudDetection"
                             )
    plt.close()
                            



2026/05/27 12:41:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/27 12:42:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/27 12:42:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'CreditCardFraudDetection'.
Created version '1' of model 'Cre

In [13]:
pipe.named_steps

{'scaler': StandardScaler(),
 'xgb_model': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=0.2, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
               max_leaves=None, min_child_weight=10, missing=nan,
               monotone_constraints=None, multi_strategy=None, n_estimators=500,
               n_jobs=None, num_parallel_tree=None, ...)}

from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV

def custom_logging_metric(y_true,y_pred):
    cm = confusion_matrix(y_true,y_pred)
    fp = cm[0,1]
    fn = cm[1,0]

    if fn == 0:
        return 0
    return -(fp + 5 * fn)

custom_score = make_scorer(custom_logging_metric,greater_is_better=False)

param_grid = {
    'xgb_model__learning_rate': [0.1,0.15,0.2],
    'xgb_model__max_depth' : [3,5,7],
    'xgb_model__n_estimators' : [300,600,800]
}

mlflow.autolog()

mlflow.set_experiment("CC_fraud_detection_auto_best_model")
with mlflow.start_run(run_name="automatic_best"):

    grid_search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring=custom_score,
        cv=3
    )

    grid_search.fit(X_train_bal,y_train_bal)

    print("The best settings were: ", grid_search.best_params_)

